In [1]:
import os
os.chdir(os.path.join(os.path.dirname("__file__"), "../.."))

In [8]:
# os.listdir()

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformer_lens
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import tqdm
from functools import partial

/Users/samswitz/miniforge3/envs/sae/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from mechint.sae import SparseAutoEncoder

In [4]:
model = transformer_lens.HookedTransformer.from_pretrained("pythia-70m")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-70m into HookedTransformer


In [5]:
EXPANSION = 4
MODEL_PATH = f"saved_models/sae_model_{EXPANSION}x.pt"

# Load saved SAE
d = 512
m = d * EXPANSION

SAE = SparseAutoEncoder(d=d, m=m)
SAE.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
SAE.eval()

device = model.cfg.device
SAE.to(device)

SparseAutoEncoder(
  (act): ReLU()
)

In [6]:
def ablation_hook(resid, hook, feature_idx):
    f = SAE.encoder(resid)
    contr = f[..., feature_idx].unsqueeze(-1) * SAE.W_dec.data[feature_idx]
    return resid - contr

In [81]:
# text = "The governor of New York announced legislation to improve public transportation in the state."

# text = "The grass in the forest"
text = "The grass"

# text = "The grass in the forest"
# text = "The land"

In [82]:
IDX = 1274

ablated_logits = model.run_with_hooks(
    text,
    fwd_hooks=[("blocks.3.hook_resid_post", partial(ablation_hook, feature_idx=IDX))]
)

In [83]:
clean_logits = model(text)

In [84]:
clean_logprobs = F.log_softmax(clean_logits.float(), dim=-1)
ablated_logprobs = F.log_softmax(ablated_logits.float(), dim=-1)
clean_probs = clean_logprobs.exp()

kl_div = (clean_probs * (clean_logprobs - ablated_logprobs)).sum(dim=-1)
print(f"KL divergence per position: {kl_div.squeeze()}")
print(f"Mean KL divergence: {kl_div.mean().item():.6f}")

clean_top = clean_logits.argmax(dim=-1).squeeze()
ablated_top = ablated_logits.argmax(dim=-1).squeeze()

tokenizer = model.tokenizer
for pos in range(len(clean_top)):
    if clean_top[pos] != ablated_top[pos]:
        print(f"Position {pos}: {tokenizer.decode(clean_top[pos])} → {tokenizer.decode(ablated_top[pos])}")

KL divergence per position: tensor([0.0000, 0.0000, 0.2402], device='mps:0', grad_fn=<SqueezeBackward0>)
Mean KL divergence: 0.080068
Position 2: es → -


In [85]:
tokens = model.to_str_tokens(text)

# KL per token position
print("KL divergence per token:\n")
for pos, (tok, kl) in enumerate(zip(tokens, kl_div.squeeze())):
    print(f"{pos:2d} {tok:>15s}  KL={kl:.6f}")

# Top-5 predictions comparison
k = 5
print("\nTop-5 predictions (clean vs ablated):\n")
for pos in range(len(tokens)):
    clean_topk = clean_logits[0, pos].topk(k)
    ablated_topk = ablated_logits[0, pos].topk(k)
    print(f"Position {pos}: '{tokens[pos]}'")
    print(f"  Clean:   {[tokenizer.decode(t) for t in clean_topk.indices]}")
    print(f"  Ablated: {[tokenizer.decode(t) for t in ablated_topk.indices]}")
    print()

KL divergence per token:

 0   <|endoftext|>  KL=0.000000
 1             The  KL=0.000000
 2           grass  KL=0.240205

Top-5 predictions (clean vs ablated):

Position 0: '<|endoftext|>'
  Clean:   ['Q', 'The', '\n', 'A', '1']
  Ablated: ['Q', 'The', '\n', 'A', '1']

Position 1: 'The'
  Clean:   [' present', ' invention', ' effect', ' role', ' following']
  Ablated: [' present', ' invention', ' effect', ' role', ' following']

Position 2: ' grass'
  Clean:   ['es', '-', 'ho', ' and', ' is']
  Ablated: ['-', 'es', ' and', ' is', 'ho']



In [86]:
# Greedy generation: clean vs ablated
def generate_with_ablation(text, feature_idx, max_new_tokens=5):
    clean_tokens = [text]
    ablated_tokens = [text]

    clean_input = text
    ablated_input = text

    for _ in range(max_new_tokens):
        clean_logits = model(clean_input)
        next_clean = tokenizer.decode(clean_logits[0, -1].argmax())
        clean_input += next_clean
        clean_tokens.append(next_clean)

        ablated_logits = model.run_with_hooks(
            ablated_input,
            fwd_hooks=[("blocks.3.hook_resid_post", partial(ablation_hook, feature_idx=feature_idx))]
        )
        next_ablated = tokenizer.decode(ablated_logits[0, -1].argmax())
        ablated_input += next_ablated
        ablated_tokens.append(next_ablated)

    print(f"Clean:   {clean_input}")
    print(f"Ablated: {ablated_input}")

generate_with_ablation(text, IDX)

Clean:   The grasses are the most important
Ablated: The grass-roots movement in the
